# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [ ]:
%pip install agent-framework agent-framework-foundry azure-ai-projects azure-identity pydantic --quiet

In [1]:
import logging
import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.agent_provider import create_provider, describe_provider


<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [ ]:
provider = create_provider()
print(f"Provider configured: {describe_provider(provider)}")

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

Provider configured: OpenAI-compatible | model=qwen3.6-plus | endpoint=https://dashscope-intl.aliyuncs.com/compatible-mode/v1
Hello there! I’m Alex, your personal luxury travel concierge, and I’m absolutely thrilled to help you craft a week that perfectly marries world-class cuisine with centuries of living history. With a budget of around $2,500, we can absolutely create an elevated, immersive experience that feels both indulgent and intelligently paced.

Before sharing my top recommendation, I’ve cross-checked current booking availability and seasonal demand for 2025–2026 travel cycles. For a 7-day trip focused on food + history that comfortably fits your budget, **Lisbon, Sintra & Porto, Portugal** is currently showing excellent inventory across boutique properties, culinary experiences, and guided heritage tours—especially in the shoulder seasons. High-demand historic neighborhoods and top-tier food/wine experiences do book 2–3 months out, so we’ll want to secure reservations promp

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [3]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

Even though I couldn't pull specific data from the tool, I can certainly recommend three fantastic destinations for a culture-loving traveler that fit comfortably within a **$2,500 budget** (excluding international flights, but covering lodging, food, activities, and local transport for a 5–7 day trip). Here are my top picks:

### 1. **Mexico City, Mexico** 🇲🇽
**Why it's perfect for culture lovers:** Mexico City is a sprawling cultural powerhouse. You can explore the ancient ruins of **Teotihuacán** just outside the city, wander through the colorful neighborhood of **Coyoacán** (home to Frida Kahlo's Blue House), and visit world-class museums like the **Museo Nacional de Antropología** (one of the best in the world). The street art, traditional markets like Mercado de la Merced, and incredible regional cuisine (UNESCO-recognized!) make it an immersive cultural experience.
- **Budget breakdown:** Street food and local meals are incredibly affordable ($3–$10), boutique or mid-range hotel

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [4]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
Here are my top ranked recommendations based on your budget of $2,500/week and desire for culture and food:

**1. Barcelona, Spain — ~$2,000/week**
- **Pros:** Best time to visit is May–Jun (perfect timing). Incredible food scene (tapas, Catalan cuisine, markets like La Boqueria), rich cultural heritage (Gaudí architecture, Gothic Quarter, museums), vibrant arts and nightlife, plus beaches. Well under budget, leaving room for extra experiences.
- **Cons:** Touristy in peak summer; beach focus may distract from pure culture/food if that's your main priority.

**2. Tokyo, Japan — ~$2,500/week**
- **Pros:** Unmatched food scene (sushi, ramen, izakayas, Michelin-starred to street food). Deep cultural immersion (temples, shrines, tea ceremonies, traditional neighborhoods). Cutting-edge and traditional culture side by side.
- **Cons:** At the very top of your budget. Best time is March–April (cherry blossom season), so late May is past peak timing. Can feel overwhe

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.